# Testing mnli data processing

I want to test and make sure the data loading/parsing/formatting all works well, without calling the model for inference.
The actual inference will happen on euler.

In [1]:
from pathlib import Path
root = Path.cwd().parent.parent
data_dir = root / "data"
dolci_dir = data_dir / "dolci"
dolci_dataset = dolci_dir / "dolci_sft_500.parquet"
dolci_professions_file = dolci_dir / "dolci_sft_500_professions.parquet"
professions_file = data_dir / "occupations" / "select_professions.json"
dolci_classified_file = dolci_dir / "dolci_mnli_gender_classification.jsonl"

import pandas as pd
import ahocorasick
import datasets
from datasets import load_dataset
import json
from tqdm import tqdm
import re

c:\Users\manth\GitHub\occupational_bias_llms\env\.pixi\envs\default\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Helper functions

In [2]:
def pipe_separate_professions(professions_str):
    """Convert a pipe-separated string of professions back into a list."""
    if professions_str:
        return professions_str.split("|")
    else:
        return []

TESTING_SAMPLE_SIZE = 100

def load_labelled_dolci(testing: bool = False):
    infile = dolci_professions_file
    dataset = datasets.load_dataset("parquet", data_files=str(infile))["train"]
    if testing:
        dataset = dataset.select([i for i in list(range(TESTING_SAMPLE_SIZE))])
    return dataset

Actual main code

In [8]:
labelled_data = load_labelled_dolci()
output_path = dolci_classified_file
i = 1
skipped = 0
with open(output_path, "w", encoding="utf-8") as f:
    for row in labelled_data:
        occupations = pipe_separate_professions(row['instruct_professions'])
        if not occupations:
            skipped += 1
            continue
        id = row['original_index']
        print(f"Occupations: {occupations} -- ID: {id}")
        sample = row["messages"] # -> returns a list
        print(type(sample), len(sample))
        content = ""
        for doc in sample:
            if doc["role"] == "user":
                content += doc["content"].lower()
                print(f"Content: {content[:200]}...")  # Print the first
        for occ in occupations:
            hypothesis_template = f"The {occ} in the sample text is {{}}."
            output_row = {
                "id": id,
                "doc": content[:100],  # Truncate doc for easier viewing in output
                "occupation": occ,
                "hypothesis_template": hypothesis_template
            }
            f.write(json.dumps(output_row, ensure_ascii=False) + '\n')
        break

Occupations: ['analyst'] -- ID: 3
<class 'list'> 2
Content: maximilian is a data analyst working on a complex project. he has processed 1800 records so far. every 50 records, he reviews the data for inconsistencies. if each review only finds 2 discrepancies, a...
